# Hackathon 1, statistics.

This project illustrates the statistics part of the course LEPL1109. In the first part of the project, you will study the mortality observed in Belgium. In the second part of the project, you try to predict the electricity consumption in Belgium using climate variables.

## Report content

•	Grades are granted to the members whose names are in the Jupyter notebook. If your name doesn’t appear on the top of the notebook, you’ll get a 0, even though you are in a group on Moodle.

•	The jupyter notebook must be compiled with printed results and next submitted via moodle. The absence of compiled results (or non-printed values) leads to a lower grade.

•	Do not comment your results directly into cells of code. Use instead a Markdown cell. 

•	"Dry" code or results not followed by a minimum of analysis / comments will be penalized.


## Report submission

•	Deadline, see moodle website. Submission after the deadline will not be accepted.

•	To submit your report, go to the section “Hackathon” on Moodle and the subsection “Remise Hackathon 1”. You can upload your work there. Once you are sure that it is your final version, click the button “Envoyer le devoir”. It is important that you don’t forget to click on this button ! 

•	Reports that have not been uploaded through Moodle will not be corrected.


## Names and Noma of participants:

Part. 1: Aurélie Wanson 79632400

Part. 2: Aurore Chaltin 17952301

Part. 3: Jérémy Cruysberghs 39092400

Part. 4: Anh-Tien Le 54992400

Part. 5: Titouan Willermain 65422300

Part. 6: Albéric Carrette 40962400

# Mortality and climate

In this section, we analyze the relationship between mortality, age groups, and seasons in Belgium using publicly available data from Statbel (see https://statbel.fgov.be/en/about-statbel/what-we-do/visualisations/mortality). The dataset contains the number of deaths recorded over time, disaggregated by age group and observed across different seasons. The objective is to study temporal patterns in mortality and assess the role of seasonal effects. 

## 1. Basic statistics

**1.1)** Load the dataset mortality_climate.csv. Convert the variables *season* and *CD_AGEGROUP* into a categorical variable. 

In [ ]:
#--------------------------------------------------------IMPORT DES PACKAGES POUR LE HACKATON----------------------------------------------------------
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
import scipy.stats as sc
from scipy.optimize import minimize
#------------------------------------------------------------------------------------------------------------------------------------------------------

df = pd.read_csv('mortality_climate.csv')

#Conversion des variables en variables catégoriques
df['season'] = df['season'].astype('category')
df['CD_AGEGROUP'] = df['CD_AGEGROUP'].astype('category')

**1.2)** Plot the evolution of the number of deaths (*MS_NUM_DEATH*) with respect to age group. What do you observe?

In [ ]:
df['date'] = pd.to_datetime(df['date'])
categories_age = ['0-24', '25-44', '45-64', '65-74', '75-84', '85+']
df['CD_AGEGROUP'] = pd.Categorical(df['CD_AGEGROUP'], categories=categories_age, ordered=True)

pivot = df.pivot_table(index='date', columns='CD_AGEGROUP', values='MS_NUM_DEATH', observed=False)

fig, ax = plt.subplots(figsize=(13, 6))
for group in categories_age : ax.plot(pivot.index, pivot[group], label=group, linewidth=1)
ax.set_xlabel('Date')
ax.set_ylabel('Nombre de décès par jour')
ax.set_title("Évolution du nombre de décès pour chaque tranche d'âge")
ax.legend(title="Tranche d'âge")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Comments:  
Nous pouvons en déduire 2 observations :  

* Il existe une dépendance en fonction de l'âge pour la mortalité : le nombre de décès quotidiens augmente avec l'âge.

* Les courbes montrent également un cycle annuel, avec des pics marqués en hiver et des creux en été. Ce cycle est de plus en plus marqué au fur et à mesure que l'âge augmente.

**1.3)** Do a scatter plot of the number of deaths (*MS_NUM_DEATH*) by age group.

In [ ]:
df.boxplot(column='MS_NUM_DEATH', by='CD_AGEGROUP', grid=True)
plt.title("Répartition des décès par tranche d'âge")
plt.suptitle('') 
plt.xlabel("Tranche d'âge")
plt.ylabel('Nombre de décès par jour')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

Comments: 
Nous observons que le nombre de décès quotidiens ainsi que la variabilité de celui-ci augmentent fortement avec l'âge.

## 2. Fit of distributions

**2.1)** For the 45–64 age group and the winter season only, fit both a Poisson and a Negative Binomial distribution to the variable 'MS_NUM_DEATH' using a maximum likelihood approach. Plot histograms overlaid with the fitted probability mass functions. Compare the log-likelihoods and select the most appropriate distribution.

In [ ]:
df = pd.read_csv('mortality_climate.csv')

# Garder le groupe d'âge 45-64 et la saison hivernale
df_filtered = df[(df['CD_AGEGROUP'] == '45-64') & (df['season'] == 'winter')].copy()

plt.figure(figsize=(10, 6))
plt.hist(
    df_filtered["MS_NUM_DEATH"],
    bins=np.arange( # Regrouper les stats
        int(df_filtered["MS_NUM_DEATH"].min()),
        int(df_filtered["MS_NUM_DEATH"].max()) + 2, # +2 car le range ne vas pas jusqu'au dernier donc +1 et encore +1 pour avoir l'intervalle entier du dernier
    )-0.5,
    edgecolor="black", # Sinon illisible 
)
plt.xlabel("Nombre de décès par jour")
plt.ylabel("Nombre de jours")
plt.title("Décès quotidiens en hiver (45-64 ans)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

# ------------------------------------- fit Poisson MS_NUM_DEATH -------------------------------------
# Avec max likelihood

x = df_filtered["MS_NUM_DEATH"].dropna().to_numpy()
n = len(x)
lam_hat = x.mean()

poisson_log_lik = sc.poisson.logpmf(x, lam_hat).sum()
# ------------------------------------- fit Poisson (fin) -------------------------------------


# ------------------------------------- fit Negative Binomial MS_NUM_DEATH -------------------------------------
def neg_log_likelihood(params): # Négatif car on veut maximiser la log-vraisemblance, mais la fonction minimize cherche à minimiser
    r, p = params
    if r <= 0 or p <= 0 or p >= 1:
        return 1e12
    return -np.sum(sc.nbinom.logpmf(x, r, p))

# Initialisation par la méthode des moments
mean_x = lam_hat
var_x = np.var(x, ddof=1)

p0 = mean_x / var_x
r0 = (mean_x ** 2) / (var_x - mean_x)

res = minimize(neg_log_likelihood, [r0, p0], bounds=[(1e-5, None), (1e-5, 1 - 1e-5)])
r_hat, p_hat = res.x # On récupère les valeurs
nb_log_lik = -res.fun
# ------------------------------------- fit Negative Binomial (fin) -------------------------------------


# ------------------------------------- plot -------------------------------------
k = np.arange(x.min(), x.max() +1)
plt.figure(figsize=(10,6))
plt.xlabel("Nombre de décès par jour")
plt.ylabel("Densité (nombre de jours/nombre de jours au total)")
plt.title("Ajustement de Poisson et Binomiale Négative (45-64 ans, hiver)")
# Histogrammes
plt.hist(x, bins=np.arange(x.min(), x.max() +2) -0.5, density = True, alpha = 0.5, edgecolor = "black", label = "Données")
# Fonction de masse de probabilité de Poisson ajustée
plt.plot(
    k,
    sc.poisson.pmf(k, lam_hat),
    "o-",
    color="tab:blue",
    label=f"Poisson (λ={lam_hat:.2f})",
)

# Fonction de masse de probabilité de Binomiale Négative ajustée
plt.plot(
    k,
    sc.nbinom.pmf(k, r_hat, p_hat),
    "s-",
    color="tab:orange",
    label=f"Binomiale négative (r={r_hat:.2f}, p={p_hat:.3f})",
)
plt.legend()
# ------------------------------------- plot -------------------------------------
plt.show()

Comments:  
### Calcul analytique de la log vraisemblance de la Poisson :  
Tout d'abord, nous émettons comme hypothèse que les variables aléatoires $x_{1}, ... x_{n}$ sont indépendantes, et nous approximons par une loi de Poisson : $x_{i} \sim Poisson(\lambda)$.  
Pour une distribution de Poisson, la fonction de probabilité est obtenue grâce au formulaire : 
$$P(X=x) = \frac{e^{-\lambda}.\lambda^{x}}{x!}$$

Nous faisons une approximation grâce à la méthode *maximum likelihood estimation (MLE)* : 
$$L(\lambda) = \prod_{i=1}^{n} \frac{e^{-\lambda}\,\lambda^{x_i}}{x_i!} = \frac{e^{-n\lambda}\,\lambda^{\sum_{i=1}^{n} x_i}}{\prod_{i=1}^{n} x_i!}
$$
Nous prenons ensuite le logarithme :
$$ l(\lambda) = log(L(\lambda)) = -n\lambda + (\sum_{i=1}^{n} x_i) log(\lambda) - \sum_{i=1}^{n} log(x_{i}!)$$
Nous pouvons voir que l'expression est une forme analytique, et qu'elle peut donc être dérivée pour trouver un maximum :
$$\frac{dl}{d\lambda} = -n + \frac{\sum_{i=1}^{n}x_i}{\lambda} = 0 $$
Nous trouvons ainsi l'estimation suivante : 

$ \hat{\lambda} = \bar{x} $  

### Calcul analytique de la log vraisemblance de la Binomiale Négative :  

Elle ne peut malheureusement pas être obtenue aussi facilement que celle de Poisson. En effet, cela est dû au fait que la formule de la Binomiale Négative, qui est une combinatoire, n'a pas d'équation analytique résolvable pour sa dérivée. Nous utiliserons donc la méthode numérique pour la calculer.  

### Comparaison des deux log vraisemblances
La log vraisemblance de la Binomiale Négative semble plus appropriée car plus proche des valeurs mesurées. L'estimation de la Poisson est trop haute près de la moyenne et trop basse dans les extrêmes alors que la Binomiale Négative est bien plus proche des valeurs mesurées, près de la moyenne et dans les extrêmes.

# 3. Regression

We now aim to assess whether seasonality has an impact on the number of deaths. We consider the following linear model:
$$\log(Y_i) = \beta_0 + \beta_1 X^S_{i1} + \cdots + \beta_3 X^S_{i3} + \beta_4 X^{age}_{i1} + \cdots + \beta_8 X^{age}_{i5}  + \varepsilon_i.$$

where $Y_i$ denotes the number of deaths, $X^S_{ij}$ are dummy variables representing the different seasons, $X^{age}_{ij}$ are dummy variables representing the different age groups, and $\varepsilon_i \sim N(0, \sigma^2)$. 

**3.1)** Why is it more appropriate to use a linear regression on $\log(Y)$ rather than directly on $Y$?

Comments:  
Pour effectuer une régression linéaire, il est nécessaire que la variable dépendante utilisée suive une distribution normale. Pourtant, grâce à la question "Fit of distributions", nous savons que Y a tendance à suivre une distribution de type Binomiale Négative. Dès lors, il n'est pas rigoureux d'utiliser Y tel quel pour effectuer la régression linéaire.  
Nous choisissons ainsi d'utiliser la variable "log(Y)" afin que sa distribution se rapproche plus de celle d'une loi Normale pour construire une régression linéaire.

**3.2)** Perform a linear regression of the number of deaths on the seasonal and group age covariates. Use the `OLS` function from the *statsmodels* package. Make sure to convert the variable *season* and *CD_AGEGROUP* into dummy variables before fitting the model.

Does seasonality have a significant impact on mortality? Is the mortality similar for each age group? 

In [ ]:
y = np.log(df["MS_NUM_DEATH"])
X = pd.get_dummies(df[["season", "CD_AGEGROUP"]], drop_first=True, dtype=float)
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

Comments:  
* Les saisons ont-elles un impact significatif sur la mortalité ?  
-> Oui, nous pouvons observer que les saisons ont un impact significatif mais relativement faible sur la mortalité. En effet, une variable explicative est considérée comme significative si la p-valeur de son test d'hypothèse "H0 : coef_variable_expl = 0" est < 0.05. Dans notre régression, nous prenons l'automne comme saison de référence :  
En hiver, nous observons que le nombre de décès par jour est supérieur d'environ 4% à celui en automne (exp(0.0407) = 1.04 -> 4% de plus). En été, nous observons que le nombre de décès par jour est inférieur d'environ 3% (exp(-0.0302) = 0.97 -> 3% de moins). Or, pour l'hiver et l'été, les p-valeurs sont toutes les deux inférieures à 5%. Dès lors, la mortalité en automne diffère significativement de celle de ces 2 saisons. En effet, la mortalité en été est significativement plus petite que celle en automne (coefficient < 0), et la mortalité en hiver est significativement plus grande que celle en automne (coefficient > 0). => Rejet de l'hypothèse nulle.  
Cependant, pour le printemps, la p-valeur est supérieure à 5%. Dès lors, nous ne pouvons pas conclure que la mortalité au printemps diffère significativement de celle en automne. => Non-rejet de l'hypothèse nulle.  
* La mortalité est-elle similaire pour chaque groupe d'âge ?  
-> Non, nous pouvons observer que la mortalité diffère fortement selon les catégories d'âge. Dans notre régression, nous prenons la tranche d'âge 0-24 ans comme référence :  
Nous observons que chez les 85 ans et plus, le nombre de décès par jour est environ 61 fois supérieur à celle des 0-24 ans (exp(4.1089) = 60.88 ~ 61). Or, pour cette tranche d'âge, la p-valeur est inférieure à 5%. Dès lors, la mortalité chez les 85 ans et plus est significativement plus grande que celle des 0-24 ans.


## 4. Hypothesis test 

We now investigate whether mortality decreases over time by comparing mortality in summer in 2022 and 2025. To this end, we aim to perform the following hypothesis test:

$$
H_0 : \mu_{2022} = \mu_{2025},
$$

$$
H_1 : \mu_{2022} \neq \mu_{2025}.
$$

**4.1)** Create two subsets of the original dataset corresponding to the years 2022 and 2025, respectively.

In [ ]:
# On garde les 4 premiers caractères de la date qui vaut l'année
df['year'] = df['date'].str[:4]

# On garde que les lignes avec la bonne année
df_2022 = df[df['year'] == '2022']
df_2025 = df[df['year'] == '2025']

# Vérifie les 5 premières lignes des sous-ensembles
print(df_2022.head())
print(df_2025.head())

**4.2)** For the 85+ age group, perform a Student’s t-test to compare mortality between 2022 and 2025 **in summer**. Check whether the assumptions required to apply this test are satisfied.

In [ ]:
# Echantillons des déces par jour, en été, des années correspondantes pour les 85+
X1 = df_2022[(df_2022['CD_AGEGROUP'] == '85+') & (df_2022['season'] == 'summer')]['MS_NUM_DEATH'].values
X2 = df_2025[(df_2025['CD_AGEGROUP'] == '85+') & (df_2025['season'] == 'summer')]['MS_NUM_DEATH'].values

# Taille des échantillons
n1 = len(X1)
n2 = len(X2)

# Moyenne des échantillons
Xb1 = np.mean(X1)
Xb2 = np.mean(X2)

# Ecart-type des échantillons
S1 = np.std(X1, ddof=1)
S2 = np.std(X2, ddof=1)

print("n1 =", n1, " moyenne =", Xb1, " écart-type =", S1)
print("n2 =", n2, " moyenne =", Xb2, " écart-type =", S2)

# Condition 1 : égalité des variances avec le test de Bartlett
Txbis, pvalbis = sc.bartlett(X1.astype(float), X2.astype(float))
print("Bartlett : statistique =", Txbis, " p-value =", pvalbis)

# Condition 2 : normalité avec le test de Shapiro-Wilk
print("Shapiro 2022 :", sc.shapiro(X1))
print("Shapiro 2025 :", sc.shapiro(X2))

# T-test de Student à la main 
Spool = np.sqrt(((n1-1)*S1**2 + (n2-1)*S2**2) / (n1 + n2 - 2))
Tx = (Xb1 - Xb2) / (Spool * np.sqrt(1/n1 + 1/n2))
alpha = 0.05
t_l = sc.t.ppf(q=alpha/2, df=n1+n2-2)
t_u = sc.t.ppf(q=1-alpha/2, df=n1+n2-2)
pval = 2 * sc.t.cdf(-np.abs(Tx), df=n1+n2-2)

print("Statistique T =", Tx)
print("Zone de non-rejet : [", t_l, ",", t_u, "]")
print("p-value =", pval)

# T-test de Student avec scipy pour vrifier
T_sc, p_sc = sc.ttest_ind(X1, X2, equal_var=True)
print("scipy :", T_sc, p_sc)


Comments:

Pour que le T-test de Student soit valable, il faut respecter ces 3 conditions :  
- L'indépendance des échantillons : les observations viennent de jours différents et de 2 années différentes, cette condition est respectée.  
- L'égalité des variances : le test de Bartlett permet de calculer l'égalité des variances, et grâce à cela, nous avons trouvé une p-valeur d'environ 0.013, ce qui est en dessous du seuil critique de 0.05, donc cette condition n'est pas respectée.  
- Normalité des données : les décès par jour de chaque échantillon doivent suivre une loi Normale. Pour cela, nous utilisons le test de Shapiro-Wilk. Nous obtenons une p-valeur de 0.0002 pour 2022, et de 1.1e-6 pour 2025. Les deux valeurs sont en dessous du seuil critique de 0.05. Cette condition n'est donc pas respectée non plus.  

Ensuite pour le T-test de Student, nous obtenons T=2.04 (arrondi), ce qui est situé hors de la zone de non rejet [-1.97 , 1.97]. Et puisque la p-valeur vaut environ 0.043, ce qui est inférieur à 0.05, nous pouvons donc rejeter l'hypothèse H0 (et garder l'hypothèse H1).

=> Nous gardons H1 : la mortalité moyenne des 85+ en été est de 129.8 décès par jour en 2022 contre 123.8 décès par jour en 2025. Il est à noter cependant que seulement une des 3 conditions est respectée, dès lors, le résultat est à prendre avec précaution.


**4.3)** Explain what the Wilcoxon test is. Why could it be relevant in our setting?

Comments: 

Le test de Wilcoxon, pour 2 échantillons indépendants, est un test non paramétrique qui, lui, ne suppose pas que les données suivent une loi Normale (contrairement au T-test Student).  
Ce test consiste à regrouper les valeurs des 2 échantillons, à les classer par ordre croissant et à leur attribuer un rang (de 1 pour la plus petite valeur jusqu'à la plus grande). Le test analyse ensuite si les rangs d'un échantillon ont tendance à être plus ou moins élevés que ceux de l'autre. Enfin, le test calcule une statistique de test et une p-valeur.

Nous avons H0 : les 2 échantillons viennent de la même distribution. (Cette hypothèse sera à rejeter si la p-valeur est inférieure à 0.05.)  
et H1 : les distributions sont différentes.

Dans notre cas, il s'agit d'une meilleure alternative du test T de Student. En effet, puisque nos données ne suivent pas une loi Normale, et que les variances ne sont pas égales, le test de Wilcoxon permet une approche plus fiable sans la normalité ni l'égalité des variances.

**4.4)** For the 85+ age group, perform a Wilcoxon test to compare mortality between 2022 and 2025 **in summer**. What do you conclude?

In [ ]:
# Classe les valeurs des deux échantillons et compare leurs rangs
U, p_mw = sc.mannwhitneyu(X1, X2, alternative='two-sided')

# Statistique U et p-value du test
print("Statistique U =", U)
print("p-value =", p_mw)

Comments:

Pour comparer la mortalité des 85+ en été entre 2022 et 2025, nous utilisons le test de Wilcoxon, plus précisemment celui de Wilcoxon-Mann-Whitney (fonction scipy.stats.mannwhitneyu) puisque les conditions du test T de Student ne sont pas respectées.

Nous avons H0 : les 2 échantillons viennent de la même distribution,
Et H1 : les distributions sont différentes.

Nous obtenons une statistique U = 5555 et une p-value d’environ 0.0023, ce qui est inférieur à 0.05. Nous pouvons donc rejeter H0 et garder H1. La mortalité en été a tendance à être différente entre 2022 et 2025.

**4.5)** Perform the Wilcoxon test but now for the 25–44 age group and the **summer**. What do you conclude in this case?

In [ ]:
# Echantillons des décès des 25-44 ans en été pour l'année 2022 et 2025
Y1 = df_2022[(df_2022['CD_AGEGROUP'] == '25-44') & (df_2022['season'] == 'summer')]['MS_NUM_DEATH'].values
Y2 = df_2025[(df_2025['CD_AGEGROUP'] == '25-44') & (df_2025['season'] == 'summer')]['MS_NUM_DEATH'].values

# Test de Wilcoxon 
U2, p_mw2 = sc.mannwhitneyu(Y1, Y2, alternative='two-sided')

print("Statistique U =", U2)
print("p-value =", p_mw2)
print("Moyenne 2022 =", np.mean(Y1), " Moyenne 2025 =", np.mean(Y2))


Comments: 

Nous utilisons le même test qu'au point précédent (Wilcoxon-Mann-Whitney), mais cette fois, pour les 25-44 ans pendant l'été, en 2022 et 2025.

Nous avons toujours H0 : les 2 échantillons viennent de la même distribution. Et H1 : les distributions sont différentes. 

Nous obtenons une statistique U = 4651.5 et une p-value d'environ 0.53, ce qui est supérieur à 0.05. Ainsi, cette fois-ci, nous ne rejetons pas H0, et nous pouvons dire que nous ne détectons pas de différence entre la mortalité des 25-44 ans en été en 2022 et celle en 2025.

# Electricity Consumption and Climate

In this section, we aim to predict electricity consumption using past consumption values and climatic variables. We rely on data provided by ELIA (see https://opendata.elia.be/explore/dataset/ods003/information/?sort=datetime), which records total electricity consumption in Belgium at a 15-minute frequency, and meteorological data from the Royal Meteorological Institute (IRM) (see https://www.meteo.be/fr/services/donnees), which provides daily average temperature observations over Belgium.

The goal is to build a predictive model for electricity demand based on its recent history and weather conditions. From a practical perspective, such forecasting is highly relevant for energy providers and grid operators, as accurately anticipating electricity demand one week ahead helps optimize production planning, ensure grid stability, and reduce operational costs.

## 5. Forecasting 

The goal is to predict electricity consumption at time $t$ using its past values and climate features. Let $Y_t$ denote electricity consumption at time $t$. We consider the following regression model:

$$Y_t = \beta_0 + \beta_1 Y_{t-7Days} + \beta_2 \bar{T}_{t-7Days} + \beta_3 \bar{T}_{t-7Days}^2 + \varepsilon_t,$$

where $Y_{t-7Days}$ denotes electricity consumption observed 7 days earlier, $\bar{T}_{t-7Days}$ is the average temperature observed 7 days earlier$^1$, and $\varepsilon_t \sim \mathcal{N}(0,\sigma^2)$ is an error term.

$^1$ The average temperature is measured at a daily frequency; therefore, $\bar{T}_{t-7Days}$ is constant for all time points $t$ within the same day.

**5.1)** Load the dataset *electricity_climate.csv* and construct the dataset required to perform the regression.

**Warning:** the data are observed at 15-minute intervals.

To build the regression dataset, each row must contain:
- electricity consumption at time $t$: $Y_t$,
- electricity consumption at time $t - 7$ days: $Y_{t-7Days}$,
- temperature at time $t - 7$ days: $\bar{T}_{t-7Days}$,
- the squared temperature at time $t - 7$ days: $\bar{T}_{t-7Days}^2$.

In [ ]:
df = pd.read_csv("electricity_climate.csv")

df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.sort_values("Datetime").drop_duplicates(subset="Datetime").reset_index(drop = True) # Trier dans l'ordre chonologique

lag = 7 * 24 * 4 # = 7 (jours) * 24 (heures) * 4 (observations/heure) = 672 observations => prédictions se font avec observations faites 7 jours avant

print(df["Datetime"].diff().value_counts())

# Instancier les variables retardées de 7 jours:

df["Y_lag"] = df["Consumption"].shift(lag)
df["T_lag"] = df["temp_avg"].shift(lag)
df["T_lag_sq"] = df["temp_avg_sq"].shift(lag)

# Créer la dataset de la régression :

df_reg = df[["Datetime", "Consumption", "Y_lag", "T_lag", "T_lag_sq"]].dropna().rename(columns={"Consumption" : "Y"}).reset_index(drop = True)  

print(df_reg.head())
print(df_reg.shape)

**5.2)** Split the dataset into a training set (first 75%) and a test set (last 25%).  Use the `OLS` function from the *statsmodels* package to fit linear regression model using only $Y_{t-7Days}$ as explanatory variable.

Analyze the regression output, discuss the statistical significance and performance of the model.

In [ ]:
# Séparer les données : training set (premiers 75%) & test set (derniers 25%)
n_lignes = len(df_reg)
n_lignes_split = int(0.75 * n_lignes)

training_set = df_reg.iloc[:n_lignes_split].copy() #Lignes de données : 0 à n_lignes_split-1
test_set = df_reg.iloc[n_lignes_split:].copy() #Lignes de données : n_lignes_split à n_lignes-1

# Définir les variables pour la régression
X_training = sm.add_constant(training_set[["Y_lag"]]) #Ajout de la constante bêta0 pour la régression
y_training = training_set["Y"]

X_test = sm.add_constant(test_set[["Y_lag"]]) #Ajout de la constante bêta0 pour la régression
y_test = test_set["Y"]

# Régression linéaire
model = sm.OLS(y_training, X_training).fit()

print(model.summary())

# Prédiction grâce aux données du training_set
y_prediction = model.predict(X_test)

Comments:  
* Analyse de la sortie de la régression :  
-> La régression peut s'écrire de la forme : estimateur(Y_t) = 1368.6576 + 0.8342.Y_lag = bêta0 + bêta1.Y_lag  
-> Nous remarquons que les 2 coefficients sont tous les deux positifs. D'une part, nous aurons tendance à avoir une valeur de Y_t grande et positive. Et d'autre part, la valeur de Y_t augmente au plus Y_lag augmente.  
* Signification statistique :  
-> Nous pouvons observer que les 2 p-valeurs associées aux tests d'hypothèses des coefficients bêta :  
H0 : bêta0 = 0 et H0 : bêta1 = 0,  
sont fortement petites (de l'ordre < 0.001). Dès lors, il est peu probable que bêta0 et bêta1 soient non-significatifs. Nous pouvons donc rejeter l'hypothèse nulle.  
-> Dans le cas du test de Fisher qui effectue le test d'hypothèse associé aux 2 coefficients bêta en même temps :  
H0 : bêta0 = bêta1 = 0,  
nous observons à nouveau qu'il serait très peu probable que notre modèle soit non-significatif car la p-valeur est de l'ordre < 0.001. Nous pouvons donc rejeter l'hypothèse nulle.  
* Performance du modèle :  
-> Pour notre régression, nous observons que R²= 68.8%, ce qui signifie que 68.8% de la variation de Y est expliqué par le modèle et que 31.2% ne sont pas explicable par celui-ci. Idem pour R² ajusté, qui prend en compte le nombre de variables explicatives et le nombre d'observations et qui pourtant vaut 68.8% également.


**5.3)** Repeat the regression by including the climate variables in the model.

Are the climate variables statistically significant in explaining electricity consumption 7 days ahead?

In [ ]:
# Variables explicatives dont les variables de climat
features = ["Y_lag", "T_lag", "T_lag_sq"]

# Définir les variables pour la régression
X_train = sm.add_constant(training_set[features])
y_train = training_set["Y"]

X_test = sm.add_constant(test_set[features])
y_test = test_set["Y"]

# Régression linéaire
model_climate = sm.OLS(y_train, X_train).fit()

print(model_climate.summary())

# Prédiction grâce aux données du training_set
y_pred_climate = model_climate.predict(X_test)



Comments:  
* Sortie de la régression : estimateur(Y_t) = 1289.4140 + 0.8354.Y_lag + 23.4912.T_lag - 1.1978.T_lag_sq = bêta0 + bêta1.Y_lag + bêta2.T_lag + bêta3.T_lag_sq  
* Signification statistique :  
-> Nous observations que les tests d'hypothèses de toutes les variables explicatives :  
H0 : bêta0 = 0; H0 : bêta1 = 0; H0 : bêta2 = 0; H0 : bêta3 = 0,  
résultent en des p-valeurs d'ordre < 0.001. Dès lors, toutes ces variables utilisées pour prévoir la consommation d'électricité 7 jours à l'avance sont significatives. Nous pouvons rejeter l'hypothèse nulle de chacun de ces coefficients.

**5.4)** Compute the MAE and RMSE on the test set for both models: with and without climate variables.

Does the inclusion of climate variables improve the predictive performance of the model?

In [ ]:
# Modèle sans climat 
# Erreur entre la vraie consommation et celle prédite 
erreur_sans = y_test - y_prediction

# Erreur absolue moyenne
MAE_sans = np.mean(np.abs(erreur_sans))

# Racine de l'erreur quadratique moyenne
RMSE_sans = np.sqrt (np.mean(erreur_sans**2))

# Modèle avec climat 
erreur_avec = y_test - y_pred_climate
MAE_avec = np.mean(np.abs(erreur_avec))
RMSE_avec = np.sqrt (np.mean(erreur_avec**2))

print ("Sans climat : MAE =", MAE_sans, "RMSE =", RMSE_sans)
print ("Avec climat : MAE =", MAE_avec, "RMSE =", RMSE_avec)

Comments:   
Les valeurs obtenues (arrondi) sont :  
Sans climat : MAE = 928 et RMSE = 1259  
Avec climat : MAE = 925 et RMSE = 1256  
Les deux erreurs diminuent lorsque nous prenons en compte les variables climatiques.  
Ainsi, en effet, ces variables améliorent la performance prédictive du modèle. Toutefois, cette différence entre les 2 modèles est très faible et ne représente même pas un demi pourcent.

**5.5)** For the model including climate variables, plot the predicted electricity consumption versus the true consumption on the test set. Then, produce a second plot focusing only on the first 7 days of the test set.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test_set["Datetime"], y_test, label="Consommation mesurée", alpha=0.8)
plt.plot(test_set["Datetime"], y_prediction, label="Consommation prédite", alpha=0.8)
plt.xlabel("Date")
plt.ylabel("Consommation d'électricité")
plt.title("Consommation d'électricité mesurée vs prédite")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

#first 7 days 
length = 7*24*4
test_set_first7 = test_set[:length].copy()
y_test_first7 = y_test[:length].copy()
y_prediction_first7 = y_prediction[:length].copy()
plt.figure(figsize=(14, 5))
plt.plot(test_set_first7["Datetime"], y_test_first7, label="Consommation mesurée", alpha=0.8)
plt.plot(test_set_first7["Datetime"], y_prediction_first7, label="Consommation prédite", alpha=0.8)
plt.xlabel("Date")
plt.ylabel("Consommation d'électricité")
plt.title("Consommation d'électricité mesurée vs prédite")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Comments:  
Nous pouvons voir que l'estimation est très bonne puisque les deux courbes sont très proches.   
Cette estimation est particulièrement bonne sur les montées en consommation mais moins bonne sur les pics de consommation dû à leur variation entre les jours. 